In [1]:
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd


import torch
import torch.nn as nn
import torch.nn.functional as F


from torch.utils.data import Dataset, DataLoader


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)


print("="*90)
print("MULTIMODAL FUSION NOTEBOOK")
print("="*90)


print("PyTorch version:")
print(torch.__version__)

MULTIMODAL FUSION NOTEBOOK
PyTorch version:
2.14.0+cpu


In [3]:
# Paths

BASE_DIR = Path(
    "../"
)


TABULAR_DIR = (
    BASE_DIR /
    "final_data" /
    "splits" /
    "tabular"
)


print("="*90)
print("TABULAR DATA LOADING")
print("="*90)


train_tabular = pd.read_csv(
    TABULAR_DIR / "train_tabular.csv"
)


val_tabular = pd.read_csv(
    TABULAR_DIR / "validation_tabular.csv"
)


test_tabular = pd.read_csv(
    TABULAR_DIR / "test_tabular.csv"
)



print(
    "Train:",
    train_tabular.shape
)


print(
    "Validation:",
    val_tabular.shape
)


print(
    "Test:",
    test_tabular.shape
)

TABULAR DATA LOADING
Train: (671, 43)
Validation: (144, 43)
Test: (144, 43)


In [5]:
# Embedding paths

EMBED_DIR = (
    BASE_DIR /
    "final_data" /
    "embeddings" /
    "xlmr"
)


print("="*90)
print("TEXT EMBEDDINGS LOADING")
print("="*90)



train_text_embeddings = np.load(
    EMBED_DIR / "train_user_embeddings.npy"
)


val_text_embeddings = np.load(
    EMBED_DIR / "val_user_embeddings.npy"
)


test_text_embeddings = np.load(
    EMBED_DIR / "test_user_embeddings.npy"
)



train_text_keys = np.load(
    EMBED_DIR / "train_user_keys.npy",
    allow_pickle=True
)


val_text_keys = np.load(
    EMBED_DIR / "val_user_keys.npy",
    allow_pickle=True
)


test_text_keys = np.load(
    EMBED_DIR / "test_user_keys.npy",
    allow_pickle=True
)



print(
    "Train text:",
    train_text_embeddings.shape
)


print(
    "Validation text:",
    val_text_embeddings.shape
)


print(
    "Test text:",
    test_text_embeddings.shape
)


print("\nSample key:")
print(train_text_keys[:5])

TEXT EMBEDDINGS LOADING
Train text: (671, 768)
Validation text: (144, 768)
Test text: (143, 768)

Sample key:
[['007mi00']
 ['0mhp037']
 ['2002kingfox']
 ['2expvzwq7llpx4p']
 ['2tosgzcfj8vtsjk']]


In [6]:
print("="*90)
print("TABULAR - TEXT ALIGNMENT")
print("="*90)


print("Train overlap:")

train_overlap = set(
    train_tabular["user_key"]
).intersection(
    set(train_text_keys.flatten())
)


print(
    len(train_overlap)
)


print("\nValidation overlap:")

val_overlap = set(
    val_tabular["user_key"]
).intersection(
    set(val_text_keys.flatten())
)


print(
    len(val_overlap)
)


print("\nTest overlap:")

test_overlap = set(
    test_tabular["user_key"]
).intersection(
    set(test_text_keys.flatten())
)


print(
    len(test_overlap)
)

TABULAR - TEXT ALIGNMENT
Train overlap:
671

Validation overlap:
144

Test overlap:
143


In [8]:
import pickle


GRAPH_DIR = (
    BASE_DIR /
    "final_data" /
    "graph"
)


print("="*90)
print("GRAPH LOADING")
print("="*90)



# Load graph data

graph_data = torch.load(
    GRAPH_DIR / "graph_data_degree_only.pt",
    weights_only=False
)



# Load node mapping

with open(
    GRAPH_DIR / "node_to_id.pkl",
    "rb"
) as f:

    node_to_id = pickle.load(f)



# Load degree features

degree_features = np.load(
    GRAPH_DIR / "degree_features.npy"
)



print(graph_data)


print(
    "Degree features:",
    degree_features.shape
)


print(
    "Nodes:",
    len(node_to_id)
)

GRAPH LOADING


c:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\env\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Data(x=[13466, 3], edge_index=[2, 569486], y=[13466], train_mask=[13466], val_mask=[13466], test_mask=[13466])
Degree features: (13466, 3)
Nodes: 13466


In [9]:
print("="*90)
print("GRAPH ALIGNMENT")
print("="*90)


# normalize node keys

node_to_id_norm = {
    str(k).lower(): v
    for k, v in node_to_id.items()
}



train_graph_match = 0
val_graph_match = 0
test_graph_match = 0



for key in train_tabular["user_key"]:

    if str(key).lower() in node_to_id_norm:
        train_graph_match += 1



for key in val_tabular["user_key"]:

    if str(key).lower() in node_to_id_norm:
        val_graph_match += 1



for key in test_tabular["user_key"]:

    if str(key).lower() in node_to_id_norm:
        test_graph_match += 1



print(
    "Train graph matches:",
    train_graph_match
)


print(
    "Validation graph matches:",
    val_graph_match
)


print(
    "Test graph matches:",
    test_graph_match
)

GRAPH ALIGNMENT
Train graph matches: 657
Validation graph matches: 138
Test graph matches: 140


In [10]:
print("="*90)
print("USER TO GRAPH NODE MAPPING")
print("="*90)



def get_node_id(user_key):

    key = str(user_key).lower()

    return node_to_id_norm.get(
        key,
        -1
    )



train_tabular["graph_node_id"] = (
    train_tabular["user_key"]
    .apply(get_node_id)
)


val_tabular["graph_node_id"] = (
    val_tabular["user_key"]
    .apply(get_node_id)
)


test_tabular["graph_node_id"] = (
    test_tabular["user_key"]
    .apply(get_node_id)
)



print(
    train_tabular[
        "graph_node_id"
    ].head()
)


print("\nMissing:")

print(
    "Train:",
    (train_tabular.graph_node_id == -1).sum()
)

print(
    "Val:",
    (val_tabular.graph_node_id == -1).sum()
)

print(
    "Test:",
    (test_tabular.graph_node_id == -1).sum()
)

USER TO GRAPH NODE MAPPING
0      146
1     6388
2    12355
3    11658
4     1319
Name: graph_node_id, dtype: int64

Missing:
Train: 14
Val: 6
Test: 4


In [20]:
class MultimodalDataset(Dataset):

    def __init__(
        self,
        tabular_df,
        text_embeddings,
        text_keys,
        graph_data
    ):

        self.samples = []


        feature_cols = (
            tabular_df
            .select_dtypes(
                include=[
                    "float64",
                    "float32",
                    "int64",
                    "int32"
                ]
            )
            .columns
            .tolist()
        )


        # remove target and internal mapping column

        feature_cols.remove(
            "label_binary"
        )


        feature_cols.remove(
            "graph_node_id"
        )



        for i, row in tabular_df.iterrows():

            user_key = str(
                row["user_key"]
            ).lower()



            text_match = np.where(
                text_keys.flatten()
                ==
                user_key
            )[0]


            if len(text_match) == 0:
                continue


            text_idx = text_match[0]


            node_id = int(
                row["graph_node_id"]
            )


            if node_id == -1:
                continue



            tabular_values = (
                row[feature_cols]
                .astype(float)
                .values
            )



            self.samples.append(
                {

                    "tabular":
                    torch.tensor(
                        tabular_values,
                        dtype=torch.float32
                    ),


                    "text":
                    torch.tensor(
                        text_embeddings[text_idx],
                        dtype=torch.float32
                    ),


                    "graph":
                    graph_data.x[node_id]
                    .float(),


                    "label":
                    torch.tensor(
                        row["label_binary"],
                        dtype=torch.long
                    )

                }
            )



    def __len__(self):

        return len(self.samples)



    def __getitem__(self, idx):

        return self.samples[idx]

In [25]:
train_dataset = MultimodalDataset(
    train_tabular,
    train_text_embeddings,
    train_text_keys,
    graph_data
)


val_dataset = MultimodalDataset(
    val_tabular,
    val_text_embeddings,
    val_text_keys,
    graph_data
)


test_dataset = MultimodalDataset(
    test_tabular,
    test_text_embeddings,
    test_text_keys,
    graph_data
)



print("="*90)
print("MULTIMODAL DATASET SIZE")
print("="*90)


print(
    "Train:",
    len(train_dataset)
)

print(
    "Validation:",
    len(val_dataset)
)

print(
    "Test:",
    len(test_dataset)
)

MULTIMODAL DATASET SIZE
Train: 657
Validation: 138
Test: 139


In [26]:
from torch.utils.data import DataLoader


BATCH_SIZE = 32


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)



print("="*90)
print("DATALOADERS")
print("="*90)


print(
    "Train batches:",
    len(train_loader)
)


print(
    "Validation batches:",
    len(val_loader)
)


print(
    "Test batches:",
    len(test_loader)
)

DATALOADERS
Train batches: 21
Validation batches: 5
Test batches: 5


In [27]:
# Check one batch

batch = next(
    iter(train_loader)
)


print("="*90)
print("MULTIMODAL BATCH CHECK")
print("="*90)


print(
    "Tabular:",
    batch["tabular"].shape
)


print(
    "Text:",
    batch["text"].shape
)


print(
    "Graph:",
    batch["graph"].shape
)


print(
    "Labels:",
    batch["label"].shape
)

MULTIMODAL BATCH CHECK
Tabular: torch.Size([32, 40])
Text: torch.Size([32, 768])
Graph: torch.Size([32, 3])
Labels: torch.Size([32])


In [28]:
class TabularEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(40, 64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )


    def forward(self, x):

        return self.encoder(x)



class TextEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(768, 128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )


    def forward(self, x):

        return self.encoder(x)



class GraphEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU(),
            nn.Dropout(0.3)
        )


    def forward(self, x):

        return self.encoder(x)

In [29]:
class MultimodalFusionClassifier(nn.Module):

    def __init__(self):

        super().__init__()


        self.tabular_encoder = TabularEncoder()

        self.text_encoder = TextEncoder()

        self.graph_encoder = GraphEncoder()



        self.classifier = nn.Sequential(

            nn.Linear(
                64 + 128 + 32,
                64
            ),

            nn.ReLU(),

            nn.Dropout(0.3),


            nn.Linear(
                64,
                2
            )
        )



    def forward(
        self,
        tabular,
        text,
        graph
    ):


        tabular_emb = (
            self.tabular_encoder(
                tabular
            )
        )


        text_emb = (
            self.text_encoder(
                text
            )
        )


        graph_emb = (
            self.graph_encoder(
                graph
            )
        )


        fused = torch.cat(
            [
                tabular_emb,
                text_emb,
                graph_emb
            ],
            dim=1
        )


        logits = (
            self.classifier(
                fused
            )
        )


        return logits

In [30]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)


model = MultimodalFusionClassifier()


model = model.to(device)



print("="*90)
print("MULTIMODAL FUSION MODEL")
print("="*90)


print(model)

MULTIMODAL FUSION MODEL
MultimodalFusionClassifier(
  (tabular_encoder): TabularEncoder(
    (encoder): Sequential(
      (0): Linear(in_features=40, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
    )
  )
  (text_encoder): TextEncoder(
    (encoder): Sequential(
      (0): Linear(in_features=768, out_features=128, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
    )
  )
  (graph_encoder): GraphEncoder(
    (encoder): Sequential(
      (0): Linear(in_features=3, out_features=32, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
    )
  )
  (classifier): Sequential(
    (0): Linear(in_features=224, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=2, bias=True)
  )
)


In [31]:
# Take one batch

batch = next(
    iter(train_loader)
)


tabular = batch["tabular"].to(device)

text = batch["text"].to(device)

graph = batch["graph"].to(device)



model.eval()


with torch.no_grad():

    output = model(
        tabular,
        text,
        graph
    )



print("="*90)
print("FORWARD PASS CHECK")
print("="*90)


print(
    "Input Tabular:",
    tabular.shape
)


print(
    "Input Text:",
    text.shape
)


print(
    "Input Graph:",
    graph.shape
)


print(
    "Output:",
    output.shape
)

FORWARD PASS CHECK
Input Tabular: torch.Size([32, 40])
Input Text: torch.Size([32, 768])
Input Graph: torch.Size([32, 3])
Output: torch.Size([32, 2])


In [32]:
# Collect training labels

train_labels = []

for batch in train_loader:
    train_labels.extend(
        batch["label"].numpy()
    )


train_labels = np.array(
    train_labels
)


class_counts = np.bincount(
    train_labels
)


class_weights = (
    len(train_labels)
    /
    (2 * class_counts)
)


class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)



print("="*90)
print("CLASS WEIGHTS")
print("="*90)

print(
    "Counts:",
    class_counts
)


print(
    "Weights:",
    class_weights
)

CLASS WEIGHTS
Counts: [528 129]
Weights: tensor([0.6222, 2.5465])


In [33]:
# Loss function

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)



# Optimizer

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)



print("="*90)
print("TRAINING CONFIGURATION")
print("="*90)


print(
    "Optimizer: AdamW"
)


print(
    "Learning rate: 1e-3"
)


print(
    "Weight decay: 1e-4"
)


print(
    "Loss: Weighted CrossEntropyLoss"
)

TRAINING CONFIGURATION
Optimizer: AdamW
Learning rate: 1e-3
Weight decay: 1e-4
Loss: Weighted CrossEntropyLoss


In [34]:
EPOCHS = 200


best_val_loss = float("inf")


history = {
    "train_loss": [],
    "val_loss": []
}


best_model_state = None



print("="*90)
print("TRAINING MULTIMODAL FUSION MODEL")
print("="*90)



for epoch in range(EPOCHS):


    model.train()

    train_loss = 0



    for batch in train_loader:


        tabular = batch["tabular"].to(device)

        text = batch["text"].to(device)

        graph = batch["graph"].to(device)

        labels = batch["label"].to(device)



        optimizer.zero_grad()



        outputs = model(
            tabular,
            text,
            graph
        )


        loss = criterion(
            outputs,
            labels
        )



        loss.backward()


        optimizer.step()



        train_loss += loss.item()



    train_loss /= len(train_loader)



    # Validation

    model.eval()

    val_loss = 0



    with torch.no_grad():

        for batch in val_loader:


            tabular = batch["tabular"].to(device)

            text = batch["text"].to(device)

            graph = batch["graph"].to(device)

            labels = batch["label"].to(device)



            outputs = model(
                tabular,
                text,
                graph
            )


            loss = criterion(
                outputs,
                labels
            )


            val_loss += loss.item()



    val_loss /= len(val_loader)



    history["train_loss"].append(
        train_loss
    )


    history["val_loss"].append(
        val_loss
    )



    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_model_state = (
            model.state_dict()
            .copy()
        )



    if (epoch + 1) % 20 == 0:

        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Train Loss: {train_loss:.4f} "
            f"Val Loss: {val_loss:.4f}"
        )



model.load_state_dict(
    best_model_state
)


print("\nTraining completed")
print("Best model restored")

TRAINING MULTIMODAL FUSION MODEL
Epoch [20/200] Train Loss: 0.6421 Val Loss: 0.5974
Epoch [40/200] Train Loss: 0.5364 Val Loss: 0.6055
Epoch [60/200] Train Loss: 0.4810 Val Loss: 0.5838
Epoch [80/200] Train Loss: 0.3867 Val Loss: 0.6556
Epoch [100/200] Train Loss: 0.3034 Val Loss: 0.9165
Epoch [120/200] Train Loss: 0.2698 Val Loss: 0.9881
Epoch [140/200] Train Loss: 0.1932 Val Loss: 1.2751
Epoch [160/200] Train Loss: 0.1869 Val Loss: 1.4198
Epoch [180/200] Train Loss: 0.1601 Val Loss: 1.7691
Epoch [200/200] Train Loss: 0.1361 Val Loss: 1.9432

Training completed
Best model restored


In [35]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)



def evaluate_multimodal(
    model,
    loader,
    split_name
):

    model.eval()


    y_true = []
    y_prob = []


    with torch.no_grad():

        for batch in loader:


            tabular = batch["tabular"].to(device)

            text = batch["text"].to(device)

            graph = batch["graph"].to(device)

            labels = batch["label"].to(device)



            outputs = model(
                tabular,
                text,
                graph
            )


            probs = torch.softmax(
                outputs,
                dim=1
            )[:,1]



            y_true.extend(
                labels.cpu().numpy()
            )


            y_prob.extend(
                probs.cpu().numpy()
            )



    y_true = np.array(y_true)

    y_prob = np.array(y_prob)


    y_pred = (
        y_prob >= 0.5
    ).astype(int)



    print("="*90)
    print(split_name)
    print("="*90)


    print(
        f"Accuracy:  {accuracy_score(y_true,y_pred):.4f}"
    )


    print(
        f"Precision: {precision_score(y_true,y_pred):.4f}"
    )


    print(
        f"Recall:    {recall_score(y_true,y_pred):.4f}"
    )


    print(
        f"F1:        {f1_score(y_true,y_pred):.4f}"
    )


    print(
        f"ROC_AUC:   {roc_auc_score(y_true,y_prob):.4f}"
    )


    print(
        f"PR_AUC:    {average_precision_score(y_true,y_prob):.4f}"
    )


    print("\nConfusion Matrix:")

    print(
        confusion_matrix(
            y_true,
            y_pred
        )
    )


    print("\nClassification Report:")

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=[
                "Human",
                "Bot"
            ]
        )
    )



    return {
        "y_true": y_true,
        "y_prob": y_prob
    }

In [36]:
val_results = evaluate_multimodal(
    model,
    val_loader,
    "Multimodal Validation"
)


test_results = evaluate_multimodal(
    model,
    test_loader,
    "Multimodal Test"
)

Multimodal Validation
Accuracy:  0.7391
Precision: 0.3784
Recall:    0.5185
F1:        0.4375
ROC_AUC:   0.7261
PR_AUC:    0.4954

Confusion Matrix:
[[88 23]
 [13 14]]

Classification Report:
              precision    recall  f1-score   support

       Human       0.87      0.79      0.83       111
         Bot       0.38      0.52      0.44        27

    accuracy                           0.74       138
   macro avg       0.62      0.66      0.63       138
weighted avg       0.77      0.74      0.75       138

Multimodal Test
Accuracy:  0.7770
Precision: 0.4333
Recall:    0.4815
F1:        0.4561
ROC_AUC:   0.7318
PR_AUC:    0.4495

Confusion Matrix:
[[95 17]
 [14 13]]

Classification Report:
              precision    recall  f1-score   support

       Human       0.87      0.85      0.86       112
         Bot       0.43      0.48      0.46        27

    accuracy                           0.78       139
   macro avg       0.65      0.66      0.66       139
weighted avg       0.79

In [37]:
from pathlib import Path
import pandas as pd


RESULTS_DIR = (
    BASE_DIR /
    "results"
)


RESULTS_DIR.mkdir(
    exist_ok=True
)



multimodal_results = pd.DataFrame(
    [
        {
            "Model": "Multimodal Fusion Validation",
            "Accuracy": 0.7391,
            "Precision": 0.3784,
            "Recall": 0.5185,
            "F1": 0.4375,
            "ROC_AUC": 0.7261,
            "PR_AUC": 0.4954
        },

        {
            "Model": "Multimodal Fusion Test",
            "Accuracy": 0.7770,
            "Precision": 0.4333,
            "Recall": 0.4815,
            "F1": 0.4561,
            "ROC_AUC": 0.7318,
            "PR_AUC": 0.4495
        }
    ]
)



multimodal_results.to_csv(
    RESULTS_DIR /
    "multimodal_fusion_results.csv",
    index=False
)



print("="*90)
print("MULTIMODAL RESULTS SAVED")
print("="*90)


print(
    RESULTS_DIR /
    "multimodal_fusion_results.csv"
)


display(
    multimodal_results
)

MULTIMODAL RESULTS SAVED
..\results\multimodal_fusion_results.csv


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Multimodal Fusion Validation,0.7391,0.3784,0.5185,0.4375,0.7261,0.4954
1,Multimodal Fusion Test,0.7770,0.4333,0.4815,0.4561,0.7318,0.4495


In [39]:
import pandas as pd
from pathlib import Path


RESULTS_DIR = (
    BASE_DIR /
    "results"
)



all_results = []



# Load existing result files

result_files = [
    "graph_baseline_results.csv",
    "gcn_tabular_results.csv",
    "multimodal_fusion_results.csv"
]



for file in result_files:

    path = RESULTS_DIR / file

    if path.exists():

        df = pd.read_csv(path)

        all_results.append(df)



comparison_df = pd.concat(
    all_results,
    ignore_index=True
)



comparison_df

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,GCN-Degree-Only Validation,0.550725,0.213115,0.481481,0.295455,0.454121,0.174515
1,GCN-Degree-Only Test,0.571429,0.163265,0.296296,0.210526,0.460177,0.188417
2,GCN-Tabular Validation,0.630400,0.323500,0.814800,0.463200,0.705000,0.336800
3,GCN-Tabular Test,0.642900,0.311500,0.703700,0.431800,0.708900,0.346100
4,Multimodal Fusion Validation,0.739130,0.378378,0.518519,0.437500,0.726059,0.495363
5,Multimodal Fusion Test,0.776978,0.433333,0.481481,0.456140,0.731812,0.449519


In [38]:
import json
import pandas as pd
import torch
from pathlib import Path


# ===============================
# Directories
# ===============================

MODEL_DIR = BASE_DIR / "models"
RESULT_DIR = BASE_DIR / "results"

MODEL_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

PRED_DIR = RESULT_DIR / "predictions"
PRED_DIR.mkdir(exist_ok=True)



# ===============================
# 1) Save Model
# ===============================

torch.save(
    model.state_dict(),
    MODEL_DIR / "multimodal_fusion_best.pt"
)



# ===============================
# 2) Save Config
# ===============================

config = {
    "model_name": "MultimodalFusionClassifier",

    "dataset": "Gold Binary Dataset",

    "modalities": [
        "tabular",
        "text",
        "graph"
    ],

    "train_samples": len(train_dataset),
    "validation_samples": len(val_dataset),
    "test_samples": len(test_dataset),

    "tabular_features": 40,

    "text_encoder": "XLM-R",
    "text_dimension": 768,

    "graph_features": 3,

    "fusion_dimension": 224,

    "classifier": "224-64-2",

    "optimizer": "AdamW",
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,

    "loss": "Weighted CrossEntropyLoss"
}



with open(
    MODEL_DIR / "multimodal_fusion_config.json",
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )



# ===============================
# 3) Save Training History
# ===============================

history_df = pd.DataFrame(history)


history_df.to_csv(
    RESULT_DIR / "multimodal_fusion_history.csv",
    index=False
)



# ===============================
# 4) Save Predictions
# ===============================

def save_predictions(results, name):

    df = pd.DataFrame({

        "true_label":
        results["y_true"],

        "bot_probability":
        results["y_prob"],

        "predicted_label":
        (
            results["y_prob"] >= 0.5
        ).astype(int)

    })


    df.to_csv(
        PRED_DIR / name,
        index=False
    )



save_predictions(
    val_results,
    "multimodal_validation_predictions.csv"
)


save_predictions(
    test_results,
    "multimodal_test_predictions.csv"
)



# ===============================
# 5) Save Metrics Automatically
# ===============================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)



def calculate_metrics(results, name):

    y_true = results["y_true"]

    y_prob = results["y_prob"]

    y_pred = (
        y_prob >= 0.5
    ).astype(int)


    return {

        "Model": name,

        "Accuracy":
        accuracy_score(
            y_true,
            y_pred
        ),

        "Precision":
        precision_score(
            y_true,
            y_pred
        ),

        "Recall":
        recall_score(
            y_true,
            y_pred
        ),

        "F1":
        f1_score(
            y_true,
            y_pred
        ),

        "ROC_AUC":
        roc_auc_score(
            y_true,
            y_prob
        ),

        "PR_AUC":
        average_precision_score(
            y_true,
            y_prob
        )
    }



metrics_df = pd.DataFrame(
    [
        calculate_metrics(
            val_results,
            "Multimodal Fusion Validation"
        ),

        calculate_metrics(
            test_results,
            "Multimodal Fusion Test"
        )
    ]
)



metrics_df.to_csv(
    RESULT_DIR / "multimodal_fusion_results.csv",
    index=False
)



print("="*90)
print("ALL MULTIMODAL ARTIFACTS SAVED")
print("="*90)

display(metrics_df)

ALL MULTIMODAL ARTIFACTS SAVED


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Multimodal Fusion Validation,0.739130,0.378378,0.518519,0.43750,0.726059,0.495363
1,Multimodal Fusion Test,0.776978,0.433333,0.481481,0.45614,0.731812,0.449519
